# Шаг 1. Понимаем FAR-Trans

Сегодняшняя цель: прочитать таблицу, понять одну запись и сформулировать, что будет предсказывать модель.

Источник: [Университет Глазго](https://researchdata.gla.ac.uk/1658/).
Публикация: [Sanz-Cruzado, Droukas, McCreadie, 2024](https://arxiv.org/abs/2407.08692).

Используем `pandas` — библиотеку для чтения и обработки таблиц. Код ниже уже выполнен; его можно запускать повторно. Путь в следующей ячейке относится к этому компьютеру: при переносе ноутбука его нужно заменить.


In [1]:
import pandas as pd

data_dir = '/Users/annazhivikhina/Documents/Codex/2026-10-11/t-lab-recsys-ml-train-validation/work/data/FAR-Trans'

transactions = pd.read_csv(
    data_dir + "/transactions.csv",
    dtype={"customerID": "string", "ISIN": "string"},
)

print(transactions.head(3).to_string(index=False))


          customerID         ISIN  transactionID transactionType  timestamp  totalValue  units          channel marketID
00017496858921195E5A GRS434003000        7590224             Buy 2020-03-27     11000.0 5000.0 Internet Banking     XATH
00017496858921195E5A GRS434003000        7607029            Sell 2020-04-06     12080.0 5000.0 Internet Banking     XATH
00017496858921195E5A GRS434003000        7634872             Buy 2020-04-24     13400.0 5000.0 Internet Banking     XATH


`transactions` — переменная, содержащая таблицу.

- `customerID`: обезличенный идентификатор клиента. Храним как текст, чтобы сохранить начальные нули.
- `ISIN`: код финансового актива.
- `transactionType`: покупка (`Buy`) или продажа (`Sell`).
- `timestamp`: дата сделки; точного времени внутри дня здесь нет.
- `totalValue`: сумма сделки в евро в обработанном авторами наборе.
- `units`: количество единиц актива.
- `channel`: канал совершения сделки.

`head(3)` показывает первые три строки, а `to_string(index=False)` выводит их без дополнительного номера строки.

Одна строка описывает состоявшуюся сделку. Это ещё не обучающий пример «клиент — возможный актив — момент прогноза».


In [2]:
client_id = "00017496858921195E5A"

client_history = transactions[
    transactions["customerID"] == client_id
]

print(client_history[["timestamp", "transactionType", "ISIN"]])

        timestamp transactionType          ISIN
0      2020-03-27             Buy  GRS434003000
1      2020-04-06            Sell  GRS434003000
2      2020-04-24             Buy  GRS434003000
3      2020-05-07            Sell  GRS434003000
4      2020-05-15             Buy  GRS434003000
...           ...             ...           ...
34101  2022-09-30             Buy  GRS473003002
34102  2022-10-03             Buy  GRS473003002
34103  2022-10-06             Buy  GRS473003002
34104  2022-11-23            Sell  GRS473003002
34105  2022-11-24            Sell  GRS473003002

[124 rows x 3 columns]


In [4]:
client_dates = pd.to_datetime(client_history["timestamp"])

april = client_history[
    (client_dates >= "2020-04-01")
    & (client_dates < "2020-05-01")
]

print(april[["timestamp", "transactionType", "ISIN"]])

    timestamp transactionType          ISIN
1  2020-04-06            Sell  GRS434003000
2  2020-04-24             Buy  GRS434003000


In [2]:
transactions["timestamp"] = pd.to_datetime(transactions["timestamp"], errors="raise")

print("Строк:", len(transactions))
print("Клиентов:", transactions["customerID"].nunique())
print("Активов со сделками:", transactions["ISIN"].nunique())
print("Первая дата:", transactions["timestamp"].min().date())
print("Последняя дата:", transactions["timestamp"].max().date())
print("\nВиды сделок:")
print(transactions["transactionType"].value_counts().to_string())


Строк: 388048
Клиентов: 29090
Активов со сделками: 320
Первая дата: 2018-01-02
Последняя дата: 2022-11-30

Виды сделок:
transactionType
Buy     228913
Sell    159135


In [ ]:
march_buys = march[
    march["transactionType"] == "Buy"
]

march_buy_count = len(march_buys)

print("Покупок за март:", march_buy_count)

Покупок за март: 1


`pd.to_datetime` превращает текстовые даты в даты Python/pandas. `nunique()` считает разные значения, `value_counts()` — сколько раз встречается каждое значение.

Точные числа получены из скачанного файла. Они немного отличаются от таблицы в статье (388 049 записей и 321 актив); версия файлов и их проверка важны для воспроизводимости.


In [9]:
april_buys = april[
    april["transactionType"] == "Buy"
]

april_buy_count = len(april_buys)

print("Покупок за апрель:", april_buy_count)

Покупок за апрель: 1


In [10]:
asset_id = "GRS434003000"

target_buys = april_buys[
    april_buys["ISIN"] == asset_id
]

if len(target_buys) > 0:
    label = 1
else:
    label = 0

print("Правильный ответ:", label)

Правильный ответ: 1


In [11]:
example = pd.DataFrame([{
    "customerID": client_id,
    "ISIN": asset_id,
    "prediction_date": "2020-04-01",
    "user_buy_count_1m": len(march_buys),
    "label": label
}])

print(example.to_string(index=False))

          customerID         ISIN prediction_date  user_buy_count_1m  label
00017496858921195E5A GRS434003000      2020-04-01                  1      1


In [13]:
other_asset_id = "GRF000014008"

other_target_buys = april_buys[
    april_buys["ISIN"] == other_asset_id
]

print(other_target_buys)

if len(other_target_buys) > 0:
    other_label = 1
else:
    other_label = 0

print("Ответ для второго актива:", other_label)

Empty DataFrame
Columns: [customerID, ISIN, transactionID, transactionType, timestamp, totalValue, units, channel, marketID]
Index: []
Ответ для второго актива: 0


In [14]:
other_example = example.copy()
other_example["ISIN"] = other_asset_id
other_example["label"] = other_label

examples = pd.concat(
    [example, other_example],
    ignore_index=True
)

print(examples.to_string(index=False))

          customerID         ISIN prediction_date  user_buy_count_1m  label
00017496858921195E5A GRS434003000      2020-04-01                  1      1
00017496858921195E5A GRF000014008      2020-04-01                  1      0


In [16]:
asset_buy_counts = []

for isin in examples["ISIN"]:
    past_asset_buys = march_buys[
        march_buys["ISIN"] == isin
    ]
    asset_buy_counts.append(len(past_asset_buys))

examples["user_item_buy_count_1m"] = asset_buy_counts

print(
    examples[
        ["ISIN", "user_item_buy_count_1m", "label"]
    ].to_string(index=False)
)

        ISIN  user_item_buy_count_1m  label
GRS434003000                       1      1
GRF000014008                       0      0


In [3]:
assets = pd.read_csv(data_dir + "/asset_information.csv")

first_customer = transactions.iloc[0]["customerID"]
first_asset = transactions.iloc[0]["ISIN"]

asset_example = assets[assets["ISIN"] == first_asset]
print("Информация об активе:")
print(asset_example[["ISIN", "assetName", "assetCategory", "timestamp"]].to_string(index=False))

example = transactions[
    (transactions["customerID"] == first_customer)
    & (transactions["ISIN"] == first_asset)
    & (transactions["timestamp"] >= "2020-03-01")
    & (transactions["timestamp"] < "2020-05-01")
].sort_values("timestamp")

print("\nСделки одного клиента с этим активом:")
print(example[["timestamp", "transactionType", "totalValue", "units"]].to_string(index=False))


Информация об активе:
        ISIN assetName assetCategory  timestamp
GRS434003000    PPC SA         Stock 2018-01-02

Сделки одного клиента с этим активом:
 timestamp transactionType  totalValue  units
2020-03-27             Buy     11000.0 5000.0
2020-04-06            Sell     12080.0 5000.0
2020-04-24             Buy     13400.0 5000.0


В последней ячейке `==` проверяет равенство для каждой строки таблицы. Выражение в квадратных скобках оставляет подходящие строки. Символ `&` объединяет несколько условий; каждое условие заключено в скобки. `sort_values` сортирует строки по дате. `iloc[0]` выбирает первую строку по позиции.

## Предлагаемая постановка — пока без обучения

На первое число месяца для пары «клиент — доступный актив» оцениваем, купит ли клиент этот актив в начинающемся календарном месяце. Например, прогноз от 1 апреля относится к периоду **[1 апреля, 1 мая)**.

- История: только сделки и информация, доступные **до 1 апреля**.
- Ответ `label=1`: в апреле есть хотя бы одна покупка этого актива клиентом.
- Ответ `label=0`: в апреле в данных нет такой покупки. Это не означает неприязнь, отказ от показа или непригодность инвестиции.

В показанном примере покупка 27 марта — история. Покупка 24 апреля даёт будущий ответ 1. Продажу 6 апреля нельзя использовать как признак прогноза от 1 апреля.

**Выбор на обсуждение:** месячный горизонт позволяет понятно разделить прошлое и будущее. В статье авторы используют другой протокол (шесть месяцев и новые для пользователя активы). Мы допускаем повторные покупки и не будем напрямую сравнивать собственные числа с таблицей статьи.

Следующим шагом определим клиентов и активы-кандидаты по информации, доступной на дату прогноза. Затем построим несколько примеров с ответами 0 и 1. `Buy`/`Sell` из таблицы нельзя просто переименовать в такие ответы.

## Вопрос на понимание

Для прогноза от 1 апреля можем ли мы использовать продажу 6 апреля? Объясни одной фразой.

## Что важно для честного эксперимента

- Авторы добавили часть покупок, реконструируя начальные владения по последующим продажам, и скорректировали некоторые даты. Это обработанный исследовательский набор, а не неизменённый журнал. При подготовке эксперимента нужно решить, как ограничить влияние восстановленных событий; полностью идентифицировать их без флага происхождения может быть невозможно. [Раздел 3.3.1 статьи](https://arxiv.org/html/2407.08692v1#S3.SS3.SSS1).
- `transactionID` встречается у разных клиентов. Это не основание удалять строки по этому полю: единица наблюдения требует отдельного решения. Полностью одинаковых строк в файле нет.
- Метаданные клиентов и активов меняются со временем. Для каждого прогноза выбираем запись, доступную на тот момент, а не последнюю во всей таблице.
- Временной горизонт последнего прогноза должен целиком помещаться в наблюдаемый период.
- Пока построение модели и сравнение признаков не выполнялись.
